# RealPDE Pretraining

Minimal notebook: pull repo, install deps, set config, launch trainer with accelerate.

In [ ]:
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
github_token = secrets.get_secret('GITHUB_TOKEN')

In [ ]:
# --- Clone or pull repo ---
REPO_DIR = '/kaggle/working/realpde'
REPO_URL = 'https://github.com/your-username/realpde.git'  # <-- update this

authed_url = REPO_URL.replace('https://', f'https://{github_token}@')

!if [ -d {REPO_DIR} ]; then cd {REPO_DIR} && git pull; else git clone {authed_url} {REPO_DIR}; fi

In [ ]:
%cd {REPO_DIR}

In [ ]:
!uv sync

In [ ]:
%pip install -e .

In [ ]:
# --- GPU check ---
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
import os

# =============================================================================
# CONFIG — edit these variables before running
# =============================================================================

# Kaggle dataset paths (your 'realpde' dataset contains train_sim/ train_real/ etc.)
os.environ['DATA_PATH'] = '/kaggle/input/realpde/train_sim'

# Model
os.environ['MODEL_NAME'] = 'unet'
os.environ['UNET_CHANNELS'] = '16'
os.environ['UNET_N_LAYERS'] = '2'

# Rollout config
os.environ['IN_STEP'] = '20'
os.environ['OUT_STEP'] = '20'
os.environ['INTERVAL'] = '20'

# Training
os.environ['LR'] = '1e-3'
os.environ['EPOCHS'] = '50'
os.environ['BATCH_SIZE'] = '8'

# Paths
os.environ['SAVE_DIR'] = '/kaggle/working/checkpoints'

# Wandb
os.environ['WANDB_API_KEY'] = secrets.get_secret('WANDB_API_KEY')
os.environ['WANDB_PROJECT'] = 'realpde-pretrain'

print('Config set.')

In [ ]:
mixed = 'fp16' if torch.cuda.is_available() else 'no'
!accelerate launch --mixed_precision={mixed} trainer.py

In [ ]:
# --- List saved checkpoints ---
from pathlib import Path
ckpt_dir = Path(os.environ['SAVE_DIR'])
if ckpt_dir.exists():
    for f in sorted(ckpt_dir.glob('*.pth')):
        print(f'{f.name:30s} {f.stat().st_size / 1024 / 1024:.1f} MB')
else:
    print('No checkpoints found.')